# EmpowerLens — D03 span selector (Kaggle GPU)

Trains the sentence-level span selector for seeds 42, 1337, 2024 and saves **only** `results/results_span_selector/` as the notebook output (a few MB, not the whole project).

**Before running:** Settings → **Accelerator: GPU (T4/P100)**, **Internet: On**. `src/train_span_selector.py` must already be pushed to the branch below.

The repo is cloned into `/kaggle/temp` (not saved with the output); checkpoints stay there too.

In [ ]:
# 1. Fetch secret token from Kaggle and set environment variable
from kaggle_secrets import UserSecretsClient
import os

try:
    user_secrets = UserSecretsClient()
    os.environ["HF_TOKEN"] = user_secrets.get_secret("HF_TOKEN")
    print("Hugging Face Token successfully loaded into environment.")
except Exception as e:
    print("Could not load Hugging Face token. Make sure it's attached via Add-ons > Secrets.")

# 2. Clone repo into /kaggle/temp (NOT part of the output) and install deps.
REPO_URL = "https://github.com/lumia-Qcode/EmpowerLens.git"
BRANCH   = "lumia-space"

!rm -rf /kaggle/temp/empowerlens && mkdir -p /kaggle/temp && git clone --depth 1 --branch $BRANCH $REPO_URL /kaggle/temp/empowerlens
%cd /kaggle/temp/empowerlens
!pip install -q -r requirements-transformer.txt
!ls src/train_span_selector.py results/results_sentence_label/sentence_labels_train.csv


In [ ]:
# 2. Smoke test (200 sentences, 1 epoch): confirms the model downloads and the script runs. Writes nothing to config.json.
!python -m src.train_span_selector --seed 42 --smoke
!rm -rf results/results_span_selector

In [ ]:
# 3. Real run: 3 seeds. Epoch + threshold are chosen on val; test is scored once.
for seed in (42, 1337, 2024):
    !python -m src.train_span_selector --seed $seed

In [ ]:
# 4. Copy ONLY the selector results to the Kaggle output (no repo, no checkpoints), plus a zip for one-click download.
!rm -rf /kaggle/working/*
!mkdir -p /kaggle/working/results_span_selector
!cp results/results_span_selector/* /kaggle/working/results_span_selector/
!cd /kaggle/working && zip -qr results_span_selector.zip results_span_selector
!ls -la /kaggle/working/results_span_selector
!du -sh /kaggle/working